In [15]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [16]:
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [17]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [18]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).text

    # update state
    state['outline'] = outline

    return state

In [19]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).text

    state['content'] = content

    return state

In [20]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [21]:
initial_state = {'title': 'Rise of AI in Bangladesh'}

final_state = workflow.invoke(initial_state)

print("=== OUTLINE ===")
print(final_state['outline'])

print("\n=== FULL BLOG CONTENT ===")
print(final_state['content'])

=== OUTLINE ===
Here is a detailed, SEO-optimized blog outline for the topic **"The Rise of AI in Bangladesh: Opportunities, Challenges, and the Future."**

---

# Blog Outline: The Rise of AI in Bangladesh

## I. Catchy Title Options
*   *Beyond the RMG Sector: How AI is Reshaping Bangladesh’s Future*
*   *The Rise of Artificial Intelligence in Bangladesh: Silicon Valley of South Asia?*
*   *From Digital to Intelligent: Tracking Bangladesh’s AI Revolution*

---

## II. Introduction
*   **The Hook:** Briefly paint a picture of Bangladesh’s recent tech transformation (e.g., the rapid growth of "Digital Bangladesh" into "Smart Bangladesh").
*   **The Shift:** Introduce Artificial Intelligence (AI) not as a distant futuristic concept, but as a present-day reality driving change in the country.
*   **Thesis Statement:** While still in its developmental stages, AI in Bangladesh is rapidly evolving across sectors like healthcare, agriculture, and customer service, presenting unique opportuni